# D2 Interpretability

Model_Comparison_D2 compared Logistic Regression, Random Forest, and Gradient Boosting on the D2 feature set and found the three models perform similarly. This notebook does not run another comparison — it looks inside the same three fitted models to see which original variables they actually rely on.

The training data, feature set, and model configurations are exactly the ones already selected in Model_Comparison_D2. The test set is not used here; feature importance comes only from the training fit.

## Feature set

Same `D2_Core_Hypertension_Cholesterol` feature set as Experiment D2 and Model_Comparison_D2: the 7 C features plus `CCC_80` plus `CCC_90`.

In [1]:
import pandas as pd
import numpy as np

pumf = pd.read_csv("../Data_Données/pumf_cchs.csv")

print("PUMF shape:", pumf.shape)

PUMF shape: (67079, 255)


In [2]:
model_data = pumf[pumf["CCC_05"].isin([1, 2])].copy()

model_data["target"] = (model_data["CCC_05"] == 1).astype(int)

print("Modelling population:", model_data.shape[0])
print(model_data["target"].value_counts().sort_index())

Modelling population: 66242
target
0    60248
1     5994
Name: count, dtype: int64


In [3]:
FEATURES = [
    "DHHGAGE",
    "DHH_SEX",
    "EDDVH3",
    "BMI_CLASS",
    "INCDGHH",
    "SDCDGIMM",
    "GEOGPRV",
    "CCC_80",
    "CCC_90"
]

print("Number of features:", len(FEATURES))
print(FEATURES)

Number of features: 9
['DHHGAGE', 'DHH_SEX', 'EDDVH3', 'BMI_CLASS', 'INCDGHH', 'SDCDGIMM', 'GEOGPRV', 'CCC_80', 'CCC_90']


In [4]:
# Same BMI harmonization as C, D1, D2
model_data["BMI_CLASS"] = np.nan

youth_mask = model_data["DHHGAGE"] == 1
adult_mask = model_data["DHHGAGE"].isin([2, 3, 4, 5])

model_data.loc[youth_mask, "BMI_CLASS"] = model_data.loc[youth_mask, "HWTDGWHO"]
model_data.loc[adult_mask, "BMI_CLASS"] = model_data.loc[adult_mask, "HWTDGISW"]

print(model_data["BMI_CLASS"].value_counts(dropna=False).sort_index())

BMI_CLASS
1.0    26053
2.0    37045
6.0       32
9.0     3112
Name: count, dtype: int64


In [5]:
SPECIAL_CODES = {
    "DHHGAGE": [],
    "DHH_SEX": [],
    "EDDVH3": [9],
    "BMI_CLASS": [6, 9],
    "INCDGHH": [9],
    "SDCDGIMM": [9],
    "GEOGPRV": [],
    "CCC_80": [9],
    "CCC_90": [9]
}

def apply_special_codes(df, special_codes):
    result = df.copy()

    for column, codes in special_codes.items():
        if column in result.columns:
            result[column] = result[column].replace(codes, np.nan)

    return result

clean_model_data = apply_special_codes(model_data, SPECIAL_CODES)

print("Missing values after special-code handling:")
print(clean_model_data[FEATURES].isna().sum())

Missing values after special-code handling:
DHHGAGE         0
DHH_SEX         0
EDDVH3       2276
BMI_CLASS    3144
INCDGHH       947
SDCDGIMM      835
GEOGPRV         0
CCC_80        494
CCC_90        155
dtype: int64


## Method

Same population, split, and preprocessing as Model_Comparison_D2. The same three model configurations are refit on the training data only (this is the same fit already used there, just reproduced here so the fitted models are available in this notebook). No new tuning or feature selection happens. Coefficients and importances are read directly off the fitted models.

In [6]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

train_val_idx, test_idx = train_test_split(
    model_data.index,
    test_size=0.20,
    stratify=model_data["target"],
    random_state=RANDOM_STATE
)

train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=0.20,
    stratify=model_data.loc[train_val_idx, "target"],
    random_state=RANDOM_STATE
)

X_train = clean_model_data.loc[train_idx, FEATURES]
y_train = model_data.loc[train_idx, "target"]

print("Training:", X_train.shape, y_train.shape)

Training: (42394, 9) (42394,)


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, FEATURES)
])

X_train_processed = preprocessor.fit_transform(X_train)

encoded_names = preprocessor.get_feature_names_out()
encoded_names = [name.replace("categorical__", "") for name in encoded_names]

print("Processed training shape:", X_train_processed.shape)
print("First few encoded feature names:", encoded_names[:5])

Processed training shape: (42394, 34)
First few encoded feature names: ['DHHGAGE_1.0', 'DHHGAGE_2.0', 'DHHGAGE_3.0', 'DHHGAGE_4.0', 'DHHGAGE_5.0']


In [8]:
# Map each encoded dummy column back to its original D2 variable
def original_variable(name):
    matches = [f for f in FEATURES if name.startswith(f + "_")]
    return max(matches, key=len)

encoded_variable_map = pd.DataFrame({
    "encoded_feature": encoded_names,
    "variable": [original_variable(name) for name in encoded_names]
})

print(encoded_variable_map["variable"].value_counts())

variable
GEOGPRV      11
DHHGAGE       5
INCDGHH       5
EDDVH3        3
DHH_SEX       2
BMI_CLASS     2
SDCDGIMM      2
CCC_80        2
CCC_90        2
Name: count, dtype: int64


## Fit the three selected models

Same configurations already selected in Model_Comparison_D2. Class imbalance is handled the same way: `class_weight="balanced"` for Logistic Regression and Random Forest, `sample_weight` from `compute_sample_weight("balanced", ...)` for Gradient Boosting.

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

log_reg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
log_reg.fit(X_train_processed, y_train)

random_forest = RandomForestClassifier(
    n_estimators=200, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1
)
random_forest.fit(X_train_processed, y_train)

train_sample_weight = compute_sample_weight("balanced", y_train)
gradient_boosting = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42
)
gradient_boosting.fit(X_train_processed, y_train, sample_weight=train_sample_weight)

print("All three models fitted on training data.")

All three models fitted on training data.


## 1. Logistic Regression coefficients

Each one-hot category gets its own coefficient. Categories are not dropped, so there is no single reference category being compared against — each coefficient reflects that category's contribution to the log-odds of diabetes, adjusted for the intercept and every other feature.

Coefficient magnitude is only directly comparable within an unstandardized, all-categorical encoding like this if we are comparing dummy categories to each other in general terms, not treating the size as a precise, unit-free measure of "importance". A variable with more categories can spread its influence across more coefficients, so a variable's total effect is better read from the full set of its category coefficients than from any single one.

In [10]:
log_reg_coefs = pd.DataFrame({
    "variable": encoded_variable_map["variable"],
    "category": encoded_variable_map["encoded_feature"],
    "coefficient": log_reg.coef_[0]
})

top_positive = log_reg_coefs.sort_values("coefficient", ascending=False).head(8)
top_negative = log_reg_coefs.sort_values("coefficient").head(8)

print("Strongest positive coefficients (push toward diabetes = 1):")
print(top_positive.reset_index(drop=True).round(4))

print("\nStrongest negative coefficients (push toward diabetes = 0):")
print(top_negative.reset_index(drop=True).round(4))

Strongest positive coefficients (push toward diabetes = 1):
    variable       category  coefficient
0    DHHGAGE    DHHGAGE_5.0       1.3055
1    DHHGAGE    DHHGAGE_4.0       0.7736
2     CCC_80     CCC_80_1.0       0.3898
3     CCC_90     CCC_90_1.0       0.3777
4  BMI_CLASS  BMI_CLASS_2.0       0.3404
5    GEOGPRV   GEOGPRV_10.0       0.3175
6    GEOGPRV   GEOGPRV_48.0       0.2264
7    GEOGPRV   GEOGPRV_12.0       0.1785

Strongest negative coefficients (push toward diabetes = 0):
    variable       category  coefficient
0    DHHGAGE    DHHGAGE_2.0      -1.2166
1    DHHGAGE    DHHGAGE_1.0      -0.8804
2     CCC_80     CCC_80_2.0      -0.5339
3     CCC_90     CCC_90_2.0      -0.5218
4  BMI_CLASS  BMI_CLASS_1.0      -0.4845
5    GEOGPRV   GEOGPRV_60.0      -0.3106
6    DHH_SEX    DHH_SEX_2.0      -0.2421
7    GEOGPRV   GEOGPRV_47.0      -0.2218


In [11]:
log_reg_abs_by_variable = (
    log_reg_coefs.assign(abs_coefficient=log_reg_coefs["coefficient"].abs())
    .groupby("variable")["abs_coefficient"]
    .sum()
    .sort_values(ascending=False)
)

print("Sum of absolute coefficients by original variable:")
print(log_reg_abs_by_variable.round(4))

Sum of absolute coefficients by original variable:
variable
DHHGAGE      4.3023
GEOGPRV      1.8935
CCC_80       0.9237
CCC_90       0.8995
BMI_CLASS    0.8248
INCDGHH      0.3740
DHH_SEX      0.3401
SDCDGIMM     0.2932
EDDVH3       0.2732
Name: abs_coefficient, dtype: float64


## 2. Random Forest importance

`feature_importances_` gives one value per one-hot column, based on how much each split on that column reduces impurity, averaged over all trees. Importances are aggregated by summing across the dummy columns belonging to the same original variable, since importance values are non-negative and this gives a fair total contribution per variable.

In [12]:
rf_importance = pd.DataFrame({
    "variable": encoded_variable_map["variable"],
    "category": encoded_variable_map["encoded_feature"],
    "importance": random_forest.feature_importances_
})

rf_importance_by_variable = (
    rf_importance.groupby("variable")["importance"]
    .sum()
    .sort_values(ascending=False)
)

print("Random Forest importance by original variable:")
print(rf_importance_by_variable.round(4))

print("\nTop 8 individual encoded features:")
print(rf_importance.sort_values("importance", ascending=False).head(8).reset_index(drop=True).round(4))

Random Forest importance by original variable:
variable
CCC_80       0.2872
DHHGAGE      0.2553
CCC_90       0.2272
BMI_CLASS    0.0795
GEOGPRV      0.0413
INCDGHH      0.0394
EDDVH3       0.0295
DHH_SEX      0.0267
SDCDGIMM     0.0140
Name: importance, dtype: float64

Top 8 individual encoded features:
    variable       category  importance
0     CCC_80     CCC_80_1.0      0.1493
1     CCC_80     CCC_80_2.0      0.1380
2    DHHGAGE    DHHGAGE_5.0      0.1217
3     CCC_90     CCC_90_2.0      0.1206
4     CCC_90     CCC_90_1.0      0.1065
5    DHHGAGE    DHHGAGE_2.0      0.0627
6  BMI_CLASS  BMI_CLASS_2.0      0.0429
7  BMI_CLASS  BMI_CLASS_1.0      0.0365


## 3. Gradient Boosting importance

Same idea as Random Forest: `feature_importances_` per one-hot column, aggregated by summing across each original variable's dummy columns.

In [13]:
gb_importance = pd.DataFrame({
    "variable": encoded_variable_map["variable"],
    "category": encoded_variable_map["encoded_feature"],
    "importance": gradient_boosting.feature_importances_
})

gb_importance_by_variable = (
    gb_importance.groupby("variable")["importance"]
    .sum()
    .sort_values(ascending=False)
)

print("Gradient Boosting importance by original variable:")
print(gb_importance_by_variable.round(4))

print("\nTop 8 individual encoded features:")
print(gb_importance.sort_values("importance", ascending=False).head(8).reset_index(drop=True).round(4))

Gradient Boosting importance by original variable:
variable
CCC_80       0.4473
DHHGAGE      0.2488
CCC_90       0.1868
BMI_CLASS    0.0584
DHH_SEX      0.0186
GEOGPRV      0.0145
INCDGHH      0.0139
EDDVH3       0.0067
SDCDGIMM     0.0052
Name: importance, dtype: float64

Top 8 individual encoded features:
    variable       category  importance
0     CCC_80     CCC_80_2.0      0.4221
1    DHHGAGE    DHHGAGE_5.0      0.1787
2     CCC_90     CCC_90_2.0      0.1426
3     CCC_90     CCC_90_1.0      0.0442
4    DHHGAGE    DHHGAGE_4.0      0.0406
5  BMI_CLASS  BMI_CLASS_1.0      0.0341
6     CCC_80     CCC_80_1.0      0.0252
7  BMI_CLASS  BMI_CLASS_2.0      0.0242


## Compact comparison across models

Logistic Regression's summed absolute coefficient is not on the same scale as the tree models' importance (one is a log-odds measure, the other is impurity-reduction share), so the numbers below should be read as three separate rankings, not a single unified score. What is comparable is each variable's *rank* within its own model.

In [14]:
comparison = pd.DataFrame({
    "Logistic Regression (abs coef sum)": log_reg_abs_by_variable,
    "Random Forest (importance sum)": rf_importance_by_variable,
    "Gradient Boosting (importance sum)": gb_importance_by_variable
})

comparison["LR_rank"] = comparison["Logistic Regression (abs coef sum)"].rank(ascending=False).astype(int)
comparison["RF_rank"] = comparison["Random Forest (importance sum)"].rank(ascending=False).astype(int)
comparison["GB_rank"] = comparison["Gradient Boosting (importance sum)"].rank(ascending=False).astype(int)

comparison = comparison.sort_values("RF_rank")

print(comparison.round(4))

           Logistic Regression (abs coef sum)  Random Forest (importance sum)  \
variable                                                                        
CCC_80                                 0.9237                          0.2872   
DHHGAGE                                4.3023                          0.2553   
CCC_90                                 0.8995                          0.2272   
BMI_CLASS                              0.8248                          0.0795   
GEOGPRV                                1.8935                          0.0413   
INCDGHH                                0.3740                          0.0394   
EDDVH3                                 0.2732                          0.0295   
DHH_SEX                                0.3401                          0.0267   
SDCDGIMM                               0.2932                          0.0140   

           Gradient Boosting (importance sum)  LR_rank  RF_rank  GB_rank  
variable                         

## Save results

In [15]:
results_to_save = comparison.reset_index().rename(columns={"index": "variable"})
results_to_save.to_csv("d2_interpretability_results.csv", index=False)

print("Saved: d2_interpretability_results.csv")
results_to_save

Saved: d2_interpretability_results.csv


,variable,Logistic Regression (abs coef sum),Random Forest (importance sum),Gradient Boosting (importance sum),LR_rank,RF_rank,GB_rank
0,CCC_80,0.923709,0.287218,0.447266,3,1,1
1,DHHGAGE,4.302298,0.255258,0.248788,1,2,2
2,CCC_90,0.899543,0.227171,0.186765,4,3,3
3,BMI_CLASS,0.824844,0.079466,0.058363,5,4,4
4,GEOGPRV,1.893473,0.041287,0.014516,2,5,6
5,INCDGHH,0.374037,0.039405,0.013855,6,6,7
6,EDDVH3,0.273208,0.029521,0.006678,9,7,8
7,DHH_SEX,0.340098,0.026685,0.018590,7,8,5
8,SDCDGIMM,0.293229,0.013990,0.005179,8,9,9


## Interpretation

**Which variables are most influential?** Age (`DHHGAGE`), hypertension (`CCC_80`), high cholesterol (`CCC_90`), and BMI class (`BMI_CLASS`) are the top four variables in all three models, though the exact order differs. Random Forest and Gradient Boosting both rank `CCC_80` first and `DHHGAGE` second; Logistic Regression ranks `DHHGAGE` first and `CCC_80` third by summed absolute coefficient. `GEOGPRV` (province) is the one clear disagreement: it ranks 2nd for Logistic Regression but only 5th-6th for the tree models. This is likely a cardinality effect — `GEOGPRV` has 11 categories, so its coefficients accumulate a large summed absolute value even though no single province coefficient is especially large (the biggest individual `GEOGPRV` coefficients, around 0.22-0.32, are smaller than the top age or comorbidity coefficients).

**Do the three models rely on broadly similar information?** Yes, on the core variables. The same four variables (age, hypertension, cholesterol, BMI class) dominate all three rankings, and the individual encoded features with the largest Random Forest and Gradient Boosting importances are the same handful of age, `CCC_80`, and `CCC_90` categories. The models mostly disagree on how much weight to give lower-cardinality demographic variables like `GEOGPRV`, not on which variables matter most.

**Are hypertension and cholesterol actually important?** Yes. `CCC_80` and `CCC_90` are in the top 4 of every model, and for the two tree-based models they are the two single most important variables (`CCC_80` importance 0.287 for Random Forest and 0.447 for Gradient Boosting; `CCC_90` importance 0.227 and 0.187). For Logistic Regression, `CCC_80` and `CCC_90` have the largest individual coefficients among predictors after age, at roughly ±0.38 to ±0.53 in log-odds.

**Does this support C -> D1 -> D2?** Yes. Adding `CCC_80` in D1 and `CCC_90` in D2 produced real gains in F1, ROC-AUC, and PR-AUC over C (see Experiment D1 and D2 notebooks), and this interpretability analysis shows those two variables are not incidental — they are consistently among the most heavily used variables by every model that was fit on this feature set.

## Limitations

- Logistic Regression coefficients and tree-based feature importances measure different things (a linear log-odds effect vs. a share of impurity reduction across splits) and are not on a common numeric scale. The comparison table above ranks each model's variables separately for that reason.
- Feature importance describes what each model actually used to make predictions on this training data, not what causes diabetes. A variable can rank high because it is a strong predictor, a proxy for something else in the data, or connected to how the survey was structured, without being a cause.
- Random Forest and Gradient Boosting importances also do not indicate the direction of the effect, only how much a variable mattered to the model's splits. Direction (does the variable push predictions up or down) can only be read from the Logistic Regression coefficients.
- All three models were fit once on the training data for this analysis, matching the fit already used in Model_Comparison_D2. No CV or resampling was done here, so these are descriptive numbers for one specific fit, not stability-checked estimates.